In [11]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer

## Récupération des données annotées

Cette partie permet de récupérer le dataset d'entraînement à partir des topics annotés manuellement.

### Une ligne par phrase

In [20]:
# Fichier contenant les phrases segmentées
df_phrases = pd.read_csv("../models/topic_modeling/bertopic/output/reviews_phrases_with_topics_final.csv")

# Fichier clusters annotés
df_clusters = pd.read_excel("../data/labelled_topics/topics_top_words_phrases_annoté.xlsx")

# Normalisation des catégories
replace_map = {
    "Qualité Produit": "qualité produit",
    "Qualité produit": "qualité produit",
    "Service Livraison": "service livraison",
    "Service livraison": "service livraison",
    "Service Client": "service client",
    "Service client": "service client",
}

def normalize_category(cat):
    if pd.isna(cat):
        return None
    cat = cat.strip().lower()
    return replace_map.get(cat, cat)

df_clusters["Catégorie"] = df_clusters["Catégorie"].apply(normalize_category)

# Fusion sur le numéro de topic
df_phrases_merged = df_phrases.merge(
    df_clusters[["Topic", "Catégorie"]],
    left_on='topics',
    right_on='Topic',
    how='left'
).drop(columns=['Topic'])

# Catégories possibles
categories = ["qualité produit", "service livraison", "service client"]

# Retirer les phrases non annotés ou avec plusieurs catégories
df_phrases_merged = df_phrases_merged[df_phrases_merged['Catégorie'].isin(categories)]


# Création colonnes one-hot binaires
for cat in categories:
    df_phrases_merged[cat] = (df_phrases_merged['Catégorie'] == cat).astype(int)

df_phrases_merged.to_csv("./../data/labelled_topics/dataset_phrases.csv", index=False)

/tmp/ipykernel_6471/139925470.py:2: DtypeWarning: Columns (4,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_phrases = pd.read_csv("../models/topic_modeling/bertopic/output/reviews_phrases_with_topics.csv")


In [21]:
df_phrases_merged

,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,...,clean_comment,clean_tokens,comment_sans_contexte,comment_id,sentence,topics,Catégorie,qualité produit,service livraison,service client
24,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,NaN,NaN,NaN,...,annulation de commande après 2 mois d ’ attent...,"['annulation', 'commande', 'mois', 'attente', ...",annulation commande mois attente geste explica...,6,annulation de commande après 2 mois d ’ attent...,186,qualité produit,1,0,0
42,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,...,arnaque j ’ ai acheté une combinaison blanche ...,"['arnaqu', 'acheter', 'combinaison', 'blanc', ...",arnaqu acheter combinaison blanc lieu dire rec...,10,je ne vous conseille pas du tout showroom privé,5,qualité produit,1,0,0
62,Ce sont des voleurs . Showroomprivé vend des I...,1,2021-06-18 00:00:00+00:00,Laurent,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,...,ce sont des voleurs . showroomprivé vend des i...,"['voleur', 'showroompriver', 'vendre', 'ipad',...",voleur showroompriver vendre ipad reconditionn...,14,ce sont des voleurs .,19,qualité produit,1,0,0
66,Ce sont des voleurs . Showroomprivé vend des I...,1,2021-06-18 00:00:00+00:00,Laurent,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,...,ce sont des voleurs . showroomprivé vend des i...,"['voleur', 'showroompriver', 'vendre', 'ipad',...",voleur showroompriver vendre ipad reconditionn...,14,sara du service client ne comprends pas mon ex...,200,service livraison,0,1,0
67,Ce sont des voleurs . Showroomprivé vend des I...,1,2021-06-18 00:00:00+00:00,Laurent,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,...,ce sont des voleurs . showroomprivé vend des i...,"['voleur', 'showroompriver', 'vendre', 'ipad',...",voleur showroompriver vendre ipad reconditionn...,14,pour tout renseignement voici mon numéro 06 27...,198,service livraison,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37236,Après des dizaines de relances par mails et té...,1,2016-01-02 00:00:00+00:00,Rodier,NaN,TrustPilot,VeePee,NaN,NaN,NaN,...,après des dizaines de relances par mails et té...,"['dizaine', 'relance', 'mail', 'téléphone', 'c...",dizaine relance mail colis commande national g...,14940,après des dizaines de relances par mails et té...,198,service livraison,0,1,0
37275,Je rencontre le même problème que le Drogo ( a...,2,2015-12-23 00:00:00+00:00,menard,NaN,TrustPilot,VeePee,NaN,NaN,NaN,...,je rencontre le même problème que le drogo ( a...,"['rencontre', 'problème', 'drogo', 'avis', 'co...",rencontre problème drogo avis copier collé rép...,14946,copier collé de la réponse à mon mail du 16/12...,186,qualité produit,1,0,0
37410,"Bonjour , Je vous fais part de ma récente expé...",2,2015-11-16 00:00:00+00:00,Aurelie,NaN,TrustPilot,VeePee,NaN,NaN,NaN,...,"bonjour , je vous fais part de ma récente expé...","['bonjour', 'part', 'récent', 'expérience', 'v...",bonjour part récent expérience vente priver cl...,14966,après vérification auprès de notre service log...,186,qualité produit,1,0,0
37458,Je lis les autres commentaires et je constate ...,1,2015-10-29 00:00:00+00:00,Lutin33,NaN,TrustPilot,VeePee,NaN,NaN,NaN,...,je lis les autres commentaires et je constate ...,"['lire', 'commentaire', 'constater', 'stupéfac...",lire commentaire constater stupéfaction être c...,14972,- demande d'annulation et de remboursement imp...,198,service livraison,0,1,0


### Une ligne par avis
L'exécution des cellules précédentes est nécessaire pour que la suivante fonctionne.

In [22]:
cols_invariantes = ["Commentaire", "star", "date", "client", "reponse", "source", "company", "ville", "maj", "date_commande", "ecart", "clean_comment"]

agg_dict = {}

# Colonnes invariantes : first
for col in cols_invariantes:
    agg_dict[col] = (col, "first")

# Colonnes de labels : max
for col in categories:
    agg_dict[col] = (col, "max")

# Aggrégation pour avoir les catégories par avis
df_avis = (
    df_phrases_merged
    .groupby("comment_id", as_index=False)
    .agg(**agg_dict)
)

df_avis.to_csv("./../data/labelled_topics/dataset_avis.csv", index=False)

In [23]:
df_avis.head()

,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,6,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,None,None,None,NaN,annulation de commande après 2 mois d ’ attent...,1,0,0
1,10,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,None,TrustPilot,ShowRoom,None,None,None,NaN,arnaque j ’ ai acheté une combinaison blanche ...,1,0,0
2,14,Ce sont des voleurs . Showroomprivé vend des I...,1,2021-06-18 00:00:00+00:00,Laurent,None,TrustPilot,ShowRoom,None,None,None,NaN,ce sont des voleurs . showroomprivé vend des i...,1,1,0
3,27,"Voleur , escroc ! ! ! ! ! Très mauvaise expéri...",1,2021-06-15 00:00:00+00:00,PIERRE MG,"Bonjour Pierre , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,None,None,None,NaN,"voleur , escroc ! ! ! ! ! très mauvaise expéri...",1,0,0
4,28,Tres mauvais service de retour . Deux semaines...,1,2021-06-15 00:00:00+00:00,Sandra Rocha,"Bonjour Sandra , Je fais suite à votre avis au...",TrustPilot,ShowRoom,None,None,None,NaN,tres mauvais service de retour . deux semaines...,1,1,0


## XGBoost

In [25]:
df_avis = pd.read_csv('./../data/labelled_topics/dataset_avis.csv')

In [26]:
TEXT_COL = "clean_comment"
LABEL_COLS = ["qualité produit", "service livraison", "service client"]

label_names = LABEL_COLS
num_labels = len(label_names)
print(label_names)

['qualité produit', 'service livraison', 'service client']


In [27]:


texts = df_avis[TEXT_COL].values
labels = df_avis[LABEL_COLS].values

texts, X_test, labels, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

#### Sélection des hyperparamètres

In [28]:
embedding_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "camembert-base",
    "paraphrase-multilingual-MiniLM-L12-v2"
]

max_depths = [3, 4, 5]

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [29]:
f1_micro_scores = []
f1_weighted_scores = []

for emb_name in embedding_models:
    print(f"\nEmbedding : {emb_name}")
    embedder = SentenceTransformer(emb_name)

    for max_depth in max_depths:
        print(f"max_depth = {max_depth}")

        f1_micro_scores = []
        f1_weighted_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(texts), 1):

            # Split
            X_train_texts = texts[train_idx]
            X_val_texts = texts[val_idx]
            y_train = labels[train_idx]
            y_val = labels[val_idx]

            # Embeddings
            X_train = embedder.encode(X_train_texts, convert_to_numpy=True, show_progress_bar=False)
            X_val = embedder.encode(X_val_texts, convert_to_numpy=True, show_progress_bar=False)

            # Modèle
            model = MultiOutputClassifier(
                XGBClassifier(
                    eval_metric="logloss",
                    n_estimators=100,
                    max_depth=max_depth,
                    learning_rate=0.1,
                    random_state=42
                )
            )

            # Entraînement
            model.fit(X_train, y_train)

            # Prédiction
            y_pred = model.predict(X_val)

            # Scores
            f1_micro_scores.append(
                f1_score(y_val, y_pred, average="micro")
            )
            f1_weighted_scores.append(
                f1_score(y_val, y_pred, average="weighted")
            )

        print(
            f"    F1 micro: {np.mean(f1_micro_scores):.4f} | "
            f"F1 weighted: {np.mean(f1_weighted_scores):.4f}"
        )



Embedding : all-MiniLM-L6-v2
max_depth = 3
    F1 micro: 0.5697 | F1 weighted: 0.5498
max_depth = 4
    F1 micro: 0.5725 | F1 weighted: 0.5544
max_depth = 5
    F1 micro: 0.5724 | F1 weighted: 0.5530

Embedding : all-mpnet-base-v2
max_depth = 3
    F1 micro: 0.5639 | F1 weighted: 0.5404
max_depth = 4
    F1 micro: 0.5634 | F1 weighted: 0.5430
max_depth = 5


No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


    F1 micro: 0.5759 | F1 weighted: 0.5518

Embedding : camembert-base
max_depth = 3
    F1 micro: 0.6082 | F1 weighted: 0.5963
max_depth = 4
    F1 micro: 0.6079 | F1 weighted: 0.5962
max_depth = 5
    F1 micro: 0.6105 | F1 weighted: 0.5987

Embedding : paraphrase-multilingual-MiniLM-L12-v2
max_depth = 3
    F1 micro: 0.5614 | F1 weighted: 0.5467
max_depth = 4
    F1 micro: 0.5637 | F1 weighted: 0.5496
max_depth = 5
    F1 micro: 0.5569 | F1 weighted: 0.5396


#### Entraînement final

In [30]:
embedder = SentenceTransformer('camembert-base')
max_depth = 5

X_train = embedder.encode(texts, convert_to_numpy=True)
X_test = embedder.encode(X_test, convert_to_numpy=True)

No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


In [31]:
# Modèle
model = MultiOutputClassifier(
    XGBClassifier(
        eval_metric="logloss",
        n_estimators=100,
        max_depth=max_depth,
        learning_rate=0.1,
        random_state=42
    )
)

# Entraînement
model.fit(X_train, labels)

# Prédictions
y_pred = model.predict(X_test)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"F1 micro     : {f1_micro:.4f}")
print(f"F1 weighted  : {f1_weighted:.4f}")

Label 'qualité produit': Accuracy = 0.716, F1-score = 0.637
Label 'service livraison': Accuracy = 0.685, F1-score = 0.681
Label 'service client': Accuracy = 0.890, F1-score = 0.423
F1 micro     : 0.6387
F1 weighted  : 0.6312


#### Evaluation sur le dataset de test

In [32]:
# Prédiction sur le test set
y_pred = model.predict(X_test)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_test, y_pred, average='micro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"F1-score micro    : {f1_micro:.3f}")
print(f"F1-score weighted : {f1_weighted:.3f}")

Label 'qualité produit': Accuracy = 0.716, F1-score = 0.637
Label 'service livraison': Accuracy = 0.685, F1-score = 0.681
Label 'service client': Accuracy = 0.890, F1-score = 0.423
F1-score micro    : 0.639
F1-score weighted : 0.631


In [33]:
print(classification_report(y_test, y_pred, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.68      0.60      0.64       226
service livraison       0.72      0.65      0.68       284
   service client       0.76      0.29      0.42        75

        micro avg       0.70      0.58      0.64       585
        macro avg       0.72      0.51      0.58       585
     weighted avg       0.71      0.58      0.63       585
      samples avg       0.57      0.59      0.57       585



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [34]:
for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,255,65
Vrai 1,90,136


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,190,72
Vrai 1,100,184


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,464,7
Vrai 1,53,22


#### Evaluation sur le dataset gold annoté manuellement

In [35]:
df_avis_gold = pd.read_csv('./../data/test_dataset/100_avis_annote.csv', sep=";")
X_gold = df_avis_gold[TEXT_COL].values
y_gold = df_avis_gold[LABEL_COLS].values
X_gold = embedder.encode(X_gold, convert_to_numpy=True)

In [36]:
# Prédiction sur le test set
y_pred = model.predict(X_gold)

# Scores par label
for i, col in enumerate(LABEL_COLS):
    acc = np.mean(y_gold[:, i] == y_pred[:, i])
    f1 = f1_score(y_gold[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_gold, y_pred, average='micro')
f1_weighted = f1_score(y_gold, y_pred, average='weighted')

print(f"F1-score micro    : {f1_micro:.3f}")
print(f"F1-score weighted : {f1_weighted:.3f}")

Label 'qualité produit': Accuracy = 0.500, F1-score = 0.138
Label 'service livraison': Accuracy = 0.610, F1-score = 0.339
Label 'service client': Accuracy = 0.710, F1-score = 0.000
F1-score micro    : 0.192
F1-score weighted : 0.144


In [37]:
print(classification_report(y_gold, y_pred, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.24      0.10      0.14        41
service livraison       0.26      0.48      0.34        21
   service client       0.00      0.00      0.00        27

        micro avg       0.25      0.16      0.19        89
        macro avg       0.17      0.19      0.16        89
     weighted avg       0.17      0.16      0.14        89
      samples avg       0.13      0.10      0.11        89



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-define

In [38]:
for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(y_gold[:, i], y_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,46,13
Vrai 1,37,4


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,51,28
Vrai 1,11,10


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,71,2
Vrai 1,27,0
